# Valider les sorties ETL en aval

Exécutez des contrôles de qualité des données après les notebooks Bronze-vers-Silver et Silver-vers-Gold. Chaque résultat est ajouté à `audit.data_quality_result` à des fins d’examen. Ce notebook enregistre uniquement les résultats ; il ne provoque pas l’échec du pipeline.

In [ ]:
run_id = "interactive"
pipeline_name = "etl"

## Exécuter et enregistrer les contrôles

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import LongType, StringType, StructField, StructType

spark.sql("CREATE SCHEMA IF NOT EXISTS audit")
results = []

def record_check(check_name, metric_value, expected_value, passed):
    results.append((
        str(run_id), str(pipeline_name), check_name, int(metric_value),
        expected_value, "Passed" if passed else "Failed"
    ))

required_tables = [
    "silver.workforce_event",
    "gold.dim_date",
    "gold.dim_cost_center",
    "gold.dim_pay_band",
    "gold.dim_worker",
    "gold.fact_workforce_event",
]

missing_tables = []
for table_name in required_tables:
    exists = spark.catalog.tableExists(table_name)
    record_check(f"table_exists:{table_name}", int(exists), "1", exists)
    if not exists:
        missing_tables.append(table_name)

if not missing_tables:
    silver = spark.table("silver.workforce_event")
    fact = spark.table("gold.fact_workforce_event")
    dim_date = spark.table("gold.dim_date")
    dim_worker = spark.table("gold.dim_worker")
    dim_pay_band = spark.table("gold.dim_pay_band")

    silver_count = silver.count()
    fact_count = fact.count()
    record_check("silver_non_empty", silver_count, "> 0", silver_count > 0)
    record_check("fact_non_empty", fact_count, "> 0", fact_count > 0)
    record_check("fact_matches_silver_count", fact_count, str(silver_count), fact_count == silver_count)

    silver_duplicate_ids = silver.groupBy("event_id").count().filter(F.col("count") > 1).count()
    fact_duplicate_ids = fact.groupBy("event_id").count().filter(F.col("count") > 1).count()
    record_check("silver_duplicate_event_ids", silver_duplicate_ids, "0", silver_duplicate_ids == 0)
    record_check("fact_duplicate_event_ids", fact_duplicate_ids, "0", fact_duplicate_ids == 0)

    unresolved_keys = fact.filter(
        F.col("worker_key").isNull()
        | F.col("pay_band_key").isNull()
        | F.col("cost_center_key").isNull()
    ).count()
    orphan_dates = fact.select("date_key").join(
        dim_date.select("date_key"), "date_key", "left_anti"
    ).count()
    record_check("unresolved_dimension_keys", unresolved_keys, "0", unresolved_keys == 0)
    record_check("unresolved_date_keys", orphan_dates, "0", orphan_dates == 0)

    duplicate_current_workers = (dim_worker.filter(F.col("is_current") == True)
        .groupBy("employee_id").count().filter(F.col("count") > 1).count())
    duplicate_current_bands = (dim_pay_band.filter(F.col("is_current") == True)
        .groupBy("classification_group", "classification_level")
        .count().filter(F.col("count") > 1).count())
    record_check("duplicate_current_workers", duplicate_current_workers, "0", duplicate_current_workers == 0)
    record_check("duplicate_current_pay_bands", duplicate_current_bands, "0", duplicate_current_bands == 0)

schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("pipeline_name", StringType(), False),
    StructField("check_name", StringType(), False),
    StructField("metric_value", LongType(), False),
    StructField("expected_value", StringType(), False),
    StructField("status", StringType(), False),
])

result_df = spark.createDataFrame(results, schema).withColumn("checked_at", F.current_timestamp())
result_df.write.format("delta").mode("append").saveAsTable("audit.data_quality_result")
result_df.orderBy("check_name").show(truncate=False)

passed_count = sum(1 for result in results if result[5] == "Passed")
print(f"Recorded {len(results)} data-quality checks: {passed_count} passed, {len(results) - passed_count} failed")